In [4]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from typing import List, Dict, Set
import json
import os

class CryptoDataFetcher:
    def __init__(self, api_key: str, output_file: str = None):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable"
        
        # Setup output file
        if output_file is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_file = f"crypto_data_{timestamp}.csv"
        
        self.output_file = output_file
        self.output_path = f"../outputs/{self.output_file}"
        self.checkpoint_file = f"../outputs/.{output_file}.checkpoint"
        
    def fetch_crypto_list(self) -> List[Dict]:
        """Fetch the complete list of cryptocurrencies"""
        url = f"{self.base_url}/cryptocurrency-list?apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            print(f"✓ Fetched {len(data)} cryptocurrencies")
            return data
        except Exception as e:
            print(f"✗ Error fetching crypto list: {e}")
            return []
    
    def fetch_quote(self, symbol: str) -> Dict:
        """Fetch detailed quote data for a specific symbol"""
        url = f"{self.base_url}/quote?symbol={symbol}&apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            
            if data and len(data) > 0:
                return data[0]
            return None
        except Exception as e:
            print(f"✗ Error fetching quote for {symbol}: {e}")
            return None
    
    def get_already_fetched_symbols(self) -> Set[str]:
        """Get list of symbols already in the output file"""
        if not os.path.exists(self.output_path):
            return set()
        
        try:
            df = pd.read_csv(self.output_path)
            if 'symbol' in df.columns:
                fetched = set(df['symbol'].unique())
                print(f"📂 Found {len(fetched)} already-fetched symbols in existing file")
                return fetched
        except Exception as e:
            print(f"⚠️  Could not read existing file: {e}")
        
        return set()
    
    def _add_derived_metrics(self, row: Dict) -> Dict:
        """Add derived statistical features to a single row"""
        
        # Volatility metrics (using day high/low as proxy)
        if 'dayHigh' in row and 'dayLow' in row and 'price' in row:
            if row['price'] and row['dayHigh'] and row['dayLow'] and row['price'] != 0:
                row['daily_range_pct'] = round((row['dayHigh'] - row['dayLow']) / row['price'] * 100, 2)
        
        # Year-to-date performance
        if 'price' in row and 'yearLow' in row:
            if row['price'] and row['yearLow'] and row['yearLow'] != 0:
                row['ytd_gain_pct'] = round((row['price'] - row['yearLow']) / row['yearLow'] * 100, 2)
        
        # Distance from averages
        if 'price' in row and 'priceAvg50' in row:
            if row['price'] and row['priceAvg50'] and row['priceAvg50'] != 0:
                row['distance_from_50ma_pct'] = round((row['price'] - row['priceAvg50']) / row['priceAvg50'] * 100, 2)
        
        if 'price' in row and 'priceAvg200' in row:
            if row['price'] and row['priceAvg200'] and row['priceAvg200'] != 0:
                row['distance_from_200ma_pct'] = round((row['price'] - row['priceAvg200']) / row['priceAvg200'] * 100, 2)
        
        # Market cap categories
        if 'marketCap' in row and row['marketCap']:
            mcap = row['marketCap']
            if mcap < 1e7:
                row['market_cap_category'] = 'Micro (<10M)'
            elif mcap < 1e8:
                row['market_cap_category'] = 'Small (10M-100M)'
            elif mcap < 1e9:
                row['market_cap_category'] = 'Mid (100M-1B)'
            elif mcap < 1e10:
                row['market_cap_category'] = 'Large (1B-10B)'
            else:
                row['market_cap_category'] = 'Mega (>10B)'
        
        # Volume to market cap ratio
        if 'volume' in row and 'marketCap' in row:
            if row['volume'] and row['marketCap'] and row['marketCap'] != 0:
                row['volume_to_mcap_ratio'] = round(row['volume'] / row['marketCap'] * 100, 2)
        
        # Data collection timestamp
        row['data_collected_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        return row
    
    def write_row_to_csv(self, row: Dict, is_first: bool = False):
        """Write a single row to CSV incrementally"""
        df = pd.DataFrame([row])
        
        # Write header only for first row and if file doesn't exist
        write_header = is_first and not os.path.exists(self.output_path)
        
        df.to_csv(self.output_path, mode='a', header=write_header, index=False)
    
    def save_checkpoint(self, processed_symbols: Set[str]):
        """Save checkpoint of processed symbols"""
        with open(self.checkpoint_file, 'w') as f:
            json.dump(list(processed_symbols), f)
    
    def load_checkpoint(self) -> Set[str]:
        """Load checkpoint of processed symbols"""
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    return set(json.load(f))
            except:
                pass
        return set()
    
    def fetch_all_crypto_data(self, sample_size: int = None, delay: float = 0.2, 
                             checkpoint_interval: int = 50) -> str:
        """
        Fetch comprehensive data for all cryptocurrencies with incremental writes
        
        Args:
            sample_size: If specified, only fetch this many cryptos (for testing)
            delay: Delay between API calls to respect rate limits
            checkpoint_interval: Save checkpoint every N symbols
        
        Returns:
            Path to output file
        """
        print("=" * 70)
        print("CRYPTOCURRENCY DATA COLLECTION (Incremental Mode)")
        print("=" * 70)
        
        # Step 1: Get list of all cryptocurrencies
        print("\n[1/4] Fetching cryptocurrency list...")
        crypto_list = self.fetch_crypto_list()
        
        if not crypto_list:
            print("No cryptocurrencies found. Exiting.")
            return None
        
        # Limit sample size if specified
        if sample_size:
            crypto_list = crypto_list[:sample_size]
            print(f"📊 Processing sample of {sample_size} cryptocurrencies")
        
        # Step 2: Check what's already been fetched
        print(f"\n[2/4] Checking for existing data...")
        already_fetched = self.get_already_fetched_symbols()
        
        # Filter out already-fetched symbols
        symbols_to_fetch = [c for c in crypto_list if c['symbol'] not in already_fetched]
        
        if not symbols_to_fetch:
            print("✓ All symbols already fetched! Nothing to do.")
            return self.output_path
        
        print(f"📋 Need to fetch: {len(symbols_to_fetch)} symbols")
        print(f"✓ Already have: {len(already_fetched)} symbols")
        print(f"🎯 Total: {len(crypto_list)} symbols")
        
        # Step 3: Fetch quote data for each cryptocurrency
        print(f"\n[3/4] Fetching and writing data incrementally...")
        print("💾 Writing to file as we go...")
        print()
        
        successful = 0
        failed = 0
        failed_symbols = []
        is_first_write = not os.path.exists(self.output_path)
        
        start_time = time.time()
        
        for idx, crypto in enumerate(symbols_to_fetch, 1):
            symbol = crypto['symbol']
            
            # Progress indicator with ETA
            if idx % 10 == 0 or idx == len(symbols_to_fetch):
                elapsed = time.time() - start_time
                rate = idx / elapsed if elapsed > 0 else 0
                remaining = len(symbols_to_fetch) - idx
                eta_seconds = remaining / rate if rate > 0 else 0
                eta_str = str(timedelta(seconds=int(eta_seconds)))
                
                print(f"Progress: {idx}/{len(symbols_to_fetch)} ({idx/len(symbols_to_fetch)*100:.1f}%) | "
                      f"Success: {successful} | Failed: {failed} | ETA: {eta_str}")
            
            # Fetch quote data
            quote_data = self.fetch_quote(symbol)
            
            if quote_data:
                # Combine list data + quote data
                combined = {**crypto, **quote_data}
                
                # Add derived metrics
                combined = self._add_derived_metrics(combined)
                
                # Write immediately to CSV
                self.write_row_to_csv(combined, is_first=is_first_write)
                is_first_write = False
                
                successful += 1
                already_fetched.add(symbol)
                
                # Periodic checkpoint
                if idx % checkpoint_interval == 0:
                    self.save_checkpoint(already_fetched)
            else:
                failed += 1
                failed_symbols.append(symbol)
            
            # Rate limiting
            time.sleep(delay)
        
        # Final checkpoint
        self.save_checkpoint(already_fetched)
        
        # Step 4: Summary
        print("\n" + "=" * 70)
        print("COLLECTION SUMMARY")
        print("=" * 70)
        print(f"✓ Successfully fetched: {successful} cryptocurrencies")
        print(f"✗ Failed to fetch: {failed} cryptocurrencies")
        if failed_symbols[:5]:
            print(f"  Examples: {', '.join(failed_symbols[:5])}")
        print(f"💾 Output file: {self.output_file}")
        print(f"📊 Total symbols in file: {len(already_fetched)}")
        print("=" * 70)
        
        return self.output_path


def main():
    # Your API key
    API_KEY = "m8TZJWQFGH7G6x2nowAqKdzDfAyakr0T"
    
    # Initialize fetcher with a specific output filename
    # This allows resuming if the script is interrupted
    fetcher = CryptoDataFetcher(
        API_KEY, 
        output_file="cryptocurrency_market_data.csv"
    )
    
    # Fetch data
    # For testing, you can use sample_size parameter: sample_size=100
    # For full dataset, remove sample_size or set to None
    filepath = fetcher.fetch_all_crypto_data(
        sample_size=None,  # Set to None for all cryptos
        delay=0.2,  # Delay between API calls
        checkpoint_interval=50  # Save checkpoint every 50 symbols
    )
    
    if filepath:
        print(f"\n✅ Data collection complete!")
        print(f"📁 File location: {filepath}")
        
        # Load and display sample
        df = pd.read_csv(filepath)
        
        print("\n" + "=" * 70)
        print("DATA PREVIEW (First 5 rows)")
        print("=" * 70)
        print(df.head().to_string())
        
        print("\n" + "=" * 70)
        print("KEY STATISTICS")
        print("=" * 70)
        print(f"Total rows: {len(df):,}")
        print(f"Total columns: {len(df.columns)}")
        
        if 'marketCap' in df.columns:
            print(f"Total Market Cap: ${df['marketCap'].sum():,.0f}")
        if 'volume' in df.columns:
            print(f"Total 24h Volume: ${df['volume'].sum():,.0f}")
        if 'price' in df.columns:
            print(f"Average Price: ${df['price'].mean():,.2f}")
            print(f"Median Price: ${df['price'].median():,.2f}")
    else:
        print("\n⚠️  No data collected.")


if __name__ == "__main__":
    main()

CRYPTOCURRENCY DATA COLLECTION (Incremental Mode)

[1/4] Fetching cryptocurrency list...
✓ Fetched 4785 cryptocurrencies

[2/4] Checking for existing data...
📂 Found 175 already-fetched symbols in existing file
📋 Need to fetch: 4610 symbols
✓ Already have: 175 symbols
🎯 Total: 4785 symbols

[3/4] Fetching and writing data incrementally...
💾 Writing to file as we go...

Progress: 10/4610 (0.2%) | Success: 9 | Failed: 0 | ETA: 1:12:59
Progress: 20/4610 (0.4%) | Success: 19 | Failed: 0 | ETA: 1:16:01
Progress: 30/4610 (0.7%) | Success: 29 | Failed: 0 | ETA: 1:17:39
Progress: 40/4610 (0.9%) | Success: 39 | Failed: 0 | ETA: 1:18:25
Progress: 50/4610 (1.1%) | Success: 49 | Failed: 0 | ETA: 1:21:31
Progress: 60/4610 (1.3%) | Success: 59 | Failed: 0 | ETA: 1:21:18
Progress: 70/4610 (1.5%) | Success: 69 | Failed: 0 | ETA: 1:20:44
Progress: 80/4610 (1.7%) | Success: 79 | Failed: 0 | ETA: 1:20:21
Progress: 90/4610 (2.0%) | Success: 89 | Failed: 0 | ETA: 1:20:13
Progress: 100/4610 (2.2%) | Success